# iSCORS-Net — Fast Runner

**Workflow:**
1. Run **Setup** — clones the repo and installs deps.
2. Run **Train** — internal learning on the test video.
3. Run **Results** — inline visualisation.
4. Run **Download** — saves `results_<VERSION>.zip` to your machine.

---

In [ ]:
VERSION = 'v2.3'
print(f'iSCORS-Net {VERSION}')

In [ ]:
import os

REPO   = 'https://github.com/breezy90126/iscors-net.git'
BRANCH = 'claude/beautiful-volta-gFUot'

if not os.path.isdir('iscors-net'):
    !git clone --depth 1 -b {BRANCH} {REPO}
else:
    !git -C iscors-net pull

os.chdir('iscors-net')
print('Working dir:', os.getcwd())

!pip install -q -r requirements.txt scipy
print('Dependencies installed.')

In [ ]:
import os

# Always regenerate — ensures new background design (near-static) is used
video_path = './data/test_synthetic_cell.tif'
if os.path.exists(video_path):
    os.remove(video_path)
    print('Removed old video.')

os.makedirs('./data', exist_ok=True)
!python utils/generate_test_video.py

In [ ]:
!python train_internal.py

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import glob, os

result_files = sorted(glob.glob('./result/*.png'))
print(f'Result images ({len(result_files)}):', [os.path.basename(f) for f in result_files])

for path in result_files:
    img = mpimg.imread(path)
    plt.figure(figsize=(10, 4))
    plt.imshow(img)
    plt.axis('off')
    plt.title(os.path.basename(path))
    plt.tight_layout()
    plt.show()

In [ ]:
import zipfile, os, glob
from google.colab import files

zip_name = f'results_{VERSION}.zip'

with zipfile.ZipFile(zip_name, 'w', zipfile.ZIP_DEFLATED) as zf:
    for pattern in [f'./result/*{VERSION}*', f'./checkpoint/*{VERSION}*']:
        for path in glob.glob(pattern):
            zf.write(path, os.path.relpath(path, '.'))
    for path in glob.glob('./result/*.png'):
        arcname = os.path.relpath(path, '.')
        if arcname not in zf.namelist():
            zf.write(path, arcname)

print(f'Created {zip_name}  ({os.path.getsize(zip_name)/1024:.1f} KB)')
files.download(zip_name)